# Retail Marketing Governance Case

Este proyecto desarrolla un caso de marketing engineering para retencion post-prueba de un nuevo producto retail. La meta no es solo predecir churn, sino convertir el score en una decision rentable mediante gobernanza tabular, auditoria detectivesca, parsimonia modelistica, CLV probabilistico y activacion tactica.

Flujo metodologico:
- Planteamiento estrategico.
- Gobernanza y contrato tabular.
- Auditoria detectivesca: faltantes, outliers, normalidad y multicolinealidad.
- Modelado parsimonioso con baseline interpretable.
- Marketing engineering: CLV futuro, umbral economico y scorecard NBA.

In [15]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import Markdown, display
from lifetimes import BetaGeoFitter, GammaGammaFitter

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR
if not (PROJECT_ROOT / 'conf').exists() and (CURRENT_DIR.parent / 'conf').exists():
    PROJECT_ROOT = CURRENT_DIR.parent

WORKSPACE_ROOT = PROJECT_ROOT.parent
FRAMEWORK_SRC = WORKSPACE_ROOT / 'banca_360_metodologia_v4' / 'src'
if str(FRAMEWORK_SRC) not in sys.path:
    sys.path.append(str(FRAMEWORK_SRC))

from banca_360_mlops.core.metodologia import (
    audit_dataset,
    audit_missingness_mechanism,
    audit_sampling_representativeness,
    calculate_vif,
    check_normality,
    handle_outliers,
    train_supervised_model,
)

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

SETTINGS_PATH = PROJECT_ROOT / 'conf' / 'settings.yaml'
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'retail_product_marketing_synthetic.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS = yaml.safe_load(SETTINGS_PATH.read_text(encoding='utf-8'))
CONTRACT = SETTINGS['case']
MISSING_BASE_COLUMNS = ['household_income', 'nps_score', 'competitor_price_gap_pct']

df = pd.read_csv(RAW_PATH, parse_dates=CONTRACT['date_cols'])
for column in MISSING_BASE_COLUMNS:
    df[f'{column}_missing_flag'] = df[column].isna().astype(int)

MODEL_FEATURES = CONTRACT['feature_cols'] + CONTRACT.get('derived_feature_cols', [])

def compare_reference_mix(dataframe, sampling_reference_mix):
    rows = []
    for dimension, expected_map in sampling_reference_mix.items():
        observed = dataframe[dimension].value_counts(normalize=True)
        for group_name, expected_share in expected_map.items():
            observed_share = float(observed.get(group_name, 0.0))
            rows.append({
                'dimension': dimension,
                'group': group_name,
                'expected_share': expected_share,
                'observed_share': observed_share,
                'gap_pp': round((observed_share - expected_share) * 100, 2),
                'abs_gap_pp': round(abs(observed_share - expected_share) * 100, 2),
                'alert': abs(observed_share - expected_share) > 0.05,
            })
    return pd.DataFrame(rows).sort_values(['alert', 'abs_gap_pp'], ascending=[False, False]).reset_index(drop=True)

def apply_calibration(model_result, raw_probability):
    payload = model_result.get('probability_calibration')
    probability = np.clip(np.asarray(raw_probability, dtype=float), 1e-6, 1 - 1e-6)
    if payload is None or not payload.get('applied'):
        return probability
    estimator = payload.get('estimator')
    method = payload.get('method')
    if estimator is None:
        return probability
    if method == 'sigmoid':
        return np.clip(estimator.predict_proba(probability.reshape(-1, 1))[:, 1], 1e-6, 1 - 1e-6)
    if method == 'isotonic':
        return np.clip(estimator.predict(probability), 1e-6, 1 - 1e-6)
    return probability

def build_threshold_report(scorecard, contact_policy, modeling_policy):
    thresholds = np.arange(
        modeling_policy['probability_threshold_grid']['start'],
        modeling_policy['probability_threshold_grid']['end'] + 0.001,
        modeling_policy['probability_threshold_grid']['step'],
    )
    max_contacts = int(len(scorecard) * float(contact_policy['max_contact_share']))
    rows = []
    for threshold in thresholds:
        candidate = scorecard.loc[scorecard['predicted_churn_probability'] >= threshold].sort_values(
            ['expected_net_value', 'predicted_churn_probability'],
            ascending=False,
        )
        candidate = candidate.loc[candidate['expected_net_value'] > 0].head(max_contacts)
        total_cost = len(candidate) * (float(contact_policy['contact_cost']) + float(contact_policy['offer_cost']))
        total_expected_net_value = float(candidate['expected_net_value'].sum())
        rows.append({
            'threshold': round(float(threshold), 2),
            'selected_customers': int(len(candidate)),
            'contact_share': round(float(len(candidate) / len(scorecard)), 4),
            'total_expected_net_value': round(total_expected_net_value, 2),
            'average_expected_net_value': round(float(candidate['expected_net_value'].mean()) if len(candidate) else 0.0, 2),
            'roi': round(float(total_expected_net_value / total_cost) if total_cost > 0 else 0.0, 4),
        })
    return pd.DataFrame(rows).sort_values(['total_expected_net_value', 'roi'], ascending=False).reset_index(drop=True)

def assign_channel(row):
    if row['preferred_channel'] in {'email', 'app_push', 'whatsapp'}:
        return row['preferred_channel']
    if row['loyalty_tier'] == 'gold':
        return 'call_center'
    return 'sms'

def assign_offer(row):
    if row['premium_pack_adoption'] == 1:
        return 'upgrade bundle premium con envio gratis'
    if row['discount_sensitivity'] >= 0.75:
        return 'cupon 15% en segunda compra'
    if row['support_tickets_90d'] >= 3:
        return 'bono de servicio y reposicion prioritaria'
    return 'pack de recompra con beneficio de fidelidad'

def assign_nba(row):
    if row['support_tickets_90d'] >= 3 or row['nps_score_missing_flag'] == 1:
        return 'recuperacion de experiencia'
    if row['discount_sensitivity'] >= 0.75:
        return 'reactivacion promocional'
    if row['premium_pack_adoption'] == 1:
        return 'upsell premium'
    return 'recordatorio de recompra'

print(f'Proyecto: {PROJECT_ROOT}')
print(f'Dataset: {RAW_PATH.name} | Filas: {len(df):,}')

Proyecto: C:\Users\yonai\Documents\Master Analytics and Visual in Big Data\Master\retail_marketing_governance_case
Dataset: retail_product_marketing_synthetic.csv | Filas: 2,600


## 1. Planteamiento Estrategico

**Por que**: sin una meta de negocio explicitamente cuantificada, el modelado tiende a optimizar metricas tecnicas desconectadas del margen real.

**Para que**: fijar desde el inicio la funcion objetivo, el target analitico y el perimetro de decision que luego gobernaran auditoria, modelado y activacion comercial.

In [2]:
strategic_frame = pd.DataFrame([
    {
        'business_goal': CONTRACT['business_goal'],
        'business_question': CONTRACT['business_question'],
        'case_target': CONTRACT['target'],
        'target_definition': CONTRACT['target_definition'],
        'analytical_success_definition': CONTRACT['analytical_success_definition'],
    }
])

target_profile = pd.DataFrame([
    {
        'n_customers': len(df),
        'analysis_window_days': int((df['snapshot_date'].max() - df['snapshot_date'].min()).days),
        'churn_rate': round(float(df[CONTRACT['target']].mean()), 4),
        'median_margin_12m': round(float(df['margin_12m'].median()), 2),
        'median_orders_12m': round(float(df['orders_12m'].median()), 2),
    }
])

display(strategic_frame)
display(target_profile)

,business_goal,business_question,case_target,target_definition,analytical_success_definition
0,maximizar el margen neto retenido tras la prue...,que clientes tienen mayor riesgo de abandonar ...,churn_120d,1 indica abandono dentro de los 120 dias poste...,reducir churn esperado y priorizar contactos c...


,n_customers,analysis_window_days,churn_rate,median_margin_12m,median_orders_12m
0,2600,211,0.2423,41.4500,4.0000


In [3]:
target_rate = float(df[CONTRACT['target']].mean())
window_days = int((df['snapshot_date'].max() - df['snapshot_date'].min()).days)
display(Markdown(
    f"### Interpretacion de Hallazgos e Implicacion Operativa\n"
    f"- El target oficial del caso es **{CONTRACT['target']}**, es decir, abandono dentro de 120 dias posteriores a la prueba del producto.\n"
    f"- La cohorte contiene **{len(df):,} clientes** y cubre **{window_days} dias**, suficiente para discutir cobertura comercial y priorizacion.\n"
    f"- El churn observado es **{target_rate:.2%}**. Operativamente, esto implica que el problema es relevante pero no extremo: hay espacio para un modelo parsimonioso que ordene riesgo sin caer en sobre-ingenieria."
))

### Interpretacion de Hallazgos e Implicacion Operativa
- El target oficial del caso es **churn_120d**, es decir, abandono dentro de 120 dias posteriores a la prueba del producto.
- La cohorte contiene **2,600 clientes** y cubre **211 dias**, suficiente para discutir cobertura comercial y priorizacion.
- El churn observado es **24.23%**. Operativamente, esto implica que el problema es relevante pero no extremo: hay espacio para un modelo parsimonioso que ordene riesgo sin caer en sobre-ingenieria.

## 2. Gobernanza y Contrato Tabular

**Por que**: si el dataset no respeta un contrato tabular externo, cualquier score comercial queda expuesto a leakage, duplicados, drift semantico y sesgos de cobertura.

**Para que**: validar integridad del dato, roles declarados en `settings.yaml`, cobertura temporal y representatividad de la muestra antes de transformar o modelar.

In [16]:
contract_table = pd.DataFrame([

    {'role': 'id_columns', 'columns': ', '.join(CONTRACT['id_columns'])},

    {'role': 'date_cols', 'columns': ', '.join(CONTRACT['date_cols'])},

    {'role': 'feature_cols', 'columns': ', '.join(CONTRACT['feature_cols'][:10]) + ' ...'},

    {'role': 'derived_feature_cols', 'columns': ', '.join(CONTRACT.get('derived_feature_cols', []))},

])



structural_audit = audit_dataset(

    df,

    target=CONTRACT['target'],

    id_columns=CONTRACT['id_columns'],

    date_columns=CONTRACT['date_cols'],

    segment_columns=CONTRACT['representativeness_dimensions'],

    verbose=False,

)

sampling_audit = audit_sampling_representativeness(

    df,

    segment_columns=CONTRACT['representativeness_dimensions'],

    target=CONTRACT['target'],

    date_column='snapshot_date',

    verbose=False,

)

reference_mix = compare_reference_mix(df, CONTRACT['sampling_reference_mix'])

tabular_standards = structural_audit['tabular_standards']



display(contract_table)

display(structural_audit['summary'])

display(tabular_standards['id_audit'])

display(tabular_standards['date_audit'])

display(sampling_audit['summary'])

display(sampling_audit['alerts'] if not sampling_audit['alerts'].empty else pd.DataFrame([{'dimension': 'none', 'hallazgo': 'sin alertas severas', 'detalle': 'la auditoria de muestreo no encontro desequilibrios criticos', 'severidad': 'info'}]))

display(reference_mix)


,role,columns
0,id_columns,customer_id
1,date_cols,"snapshot_date, first_purchase_date, last_purch..."
2,feature_cols,"region, city_tier, acquisition_channel, prefer..."
3,derived_feature_cols,"household_income_missing_flag, nps_score_missi..."


,filas,columnas,pct_duplicados_fila,columnas_constantes,columnas_con_nulos_altos,candidatos_leakage,alertas_tabulares,alertas_representatividad
0,2600,44,0.0000,0,0,0,1,0


,columna,pct_unicidad,duplicados,cumple_identificador_unico
0,customer_id,100.0000,0,True


,columna,pct_parseo_valido,patrones_detectados,requiere_normalizacion,fecha_min,fecha_max
0,snapshot_date,100.0000,1,False,2025-09-01,2026-03-31
1,first_purchase_date,100.0000,1,False,2023-03-16,2025-12-06
2,last_purchase_date,100.0000,1,False,2025-03-12,2026-01-26


,dimensiones_segmentadas,alertas_segmentales,estrategias_estratificadas,cobertura_temporal_dias
0,3,0,0,211


,dimension,hallazgo,detalle,severidad
0,none,sin alertas severas,la auditoria de muestreo no encontro desequili...,info


,dimension,group,expected_share,observed_share,gap_pp,abs_gap_pp,alert
0,region,south,0.2200,0.2923,7.2300,7.2300,True
1,acquisition_channel,store,0.3600,0.4085,4.8500,4.8500,False
2,region,north,0.1800,0.1350,-4.5000,4.5000,False
3,region,center,0.2700,0.2427,-2.7300,2.7300,False
4,acquisition_channel,paid_social,0.1800,0.1542,-2.5800,2.5800,False
5,acquisition_channel,email,0.1600,0.1385,-2.1500,2.1500,False
6,region,east,0.1600,0.1488,-1.1200,1.1200,False
7,region,west,0.1700,0.1812,1.1200,1.1200,False
8,acquisition_channel,marketplace,0.1200,0.1185,-0.1500,0.1500,False
9,acquisition_channel,organic_search,0.1800,0.1804,0.0400,0.0400,False


In [17]:
duplicate_ids = int(tabular_standards['id_audit']['duplicados'].sum())

coverage_days = int(sampling_audit['summary'].iloc[0]['cobertura_temporal_dias'])

sampling_alerts = int(len(sampling_audit['alerts'])) if not sampling_audit['alerts'].empty else 0

mix_alerts = int(reference_mix['alert'].sum())

display(Markdown(

    f"### Interpretacion de Hallazgos e Implicacion Operativa\n"

    f"- El contrato tabular esta formalizado fuera del notebook y declara ids, fechas y features. Esto reduce ambiguedad metodologica y hace auditables los cambios futuros.\n"

    f"- Los identificadores presentan **{duplicate_ids} duplicados**: la unidad de analisis queda limpia para scorecards operativos.\n"

    f"- La cobertura temporal observada es de **{coverage_days} dias**. La lectura comercial no depende de una sola fotografia puntual.\n"

    f"- La muestra acumula **{sampling_alerts} alertas de muestreo** y **{mix_alerts} desviaciones frente al mix esperado**. Implicacion: el modelo puede ser util para priorizacion, pero la generalizacion a todo el mercado exige vigilar sobrerrepresentacion por canal o region."

))


### Interpretacion de Hallazgos e Implicacion Operativa
- El contrato tabular esta formalizado fuera del notebook y declara ids, fechas y features. Esto reduce ambiguedad metodologica y hace auditables los cambios futuros.
- Los identificadores presentan **0 duplicados**: la unidad de analisis queda limpia para scorecards operativos.
- La cobertura temporal observada es de **211 dias**. La lectura comercial no depende de una sola fotografia puntual.
- La muestra acumula **0 alertas de muestreo** y **1 desviaciones frente al mix esperado**. Implicacion: el modelo puede ser util para priorizacion, pero la generalizacion a todo el mercado exige vigilar sobrerrepresentacion por canal o region.

## 3. Auditoria Detectivesca: EDA y Saneamiento

**Por que**: imputar o modelar sin entender el mecanismo de ausencia, la estructura de outliers y la estabilidad explicativa suele producir coeficientes bonitos pero poco defendibles.

**Para que**: distinguir MCAR frente a ausencia informativa, controlar extremos sin borrar senal de mercado y revisar supuestos de forma y colinealidad antes del baseline.

In [7]:
missingness_audit = audit_missingness_mechanism(df, columns=MISSING_BASE_COLUMNS, verbose=False)
missing_profile = pd.DataFrame({
    'variable': MISSING_BASE_COLUMNS,
    'missing_rate_total': [df[column].isna().mean() for column in MISSING_BASE_COLUMNS],
    'missing_rate_churn_0': [df.loc[df[CONTRACT['target']] == 0, column].isna().mean() for column in MISSING_BASE_COLUMNS],
    'missing_rate_churn_1': [df.loc[df[CONTRACT['target']] == 1, column].isna().mean() for column in MISSING_BASE_COLUMNS],
})
missing_profile['churn_gap_pp'] = ((missing_profile['missing_rate_churn_1'] - missing_profile['missing_rate_churn_0']) * 100).round(2)

winsorized_result = handle_outliers(df, columns=CONTRACT['outlier_cols'], method='winsorize', verbose=False)
winsorized_df = winsorized_result['data']
normality_report = pd.DataFrame([
    {**check_normality(winsorized_df[column], verbose=False)['shape_summary'].iloc[0].to_dict(), 'variable': column}
    for column in ['household_income', 'avg_order_value', 'margin_12m']
])
vif_result = calculate_vif(winsorized_df, columns=CONTRACT['vif_cols'], verbose=False)

display(missingness_audit['little_mcar'])
display(missing_profile)
display(winsorized_result['report'])
display(normality_report[['variable', 'consenso', 'accion_recomendada']])
display(vif_result['report'])
display(vif_result['post_mitigation_report'])

,method,statistic,degrees_of_freedom,p_value,is_mcar_compatible
0,Little MCAR aproximado,4.7648,9,0.8543,True


,variable,missing_rate_total,missing_rate_churn_0,missing_rate_churn_1,churn_gap_pp
0,household_income,0.0777,0.0726,0.0937,2.1100
1,nps_score,0.0838,0.0574,0.1667,10.9300
2,competitor_price_gap_pct,0.1696,0.1670,0.1778,1.0800


,columna,limite_inferior,limite_superior,pct_outliers_original,metodo
3,margin_12m,5.6199,313.8546,5.4200,winsorize
1,avg_order_value,12.6273,163.4240,5.3500,winsorize
2,monetary_12m,25.0000,"1,059.6768",5.0400,winsorize
0,household_income,781.6682,"7,042.7868",3.3800,winsorize


,variable,consenso,accion_recomendada
0,household_income,evidencia de no normalidad,"Aplicar Box-Cox o log, revisar colas con Ander..."
1,avg_order_value,evidencia de no normalidad,"Aplicar Box-Cox o log, revisar colas con Ander..."
2,margin_12m,evidencia de no normalidad,"Aplicar Box-Cox o log, revisar colas con Ander..."


,columna,vif,nivel_alerta
0,monetary_12m,51.8693,intervencion_critica
1,margin_12m,44.6260,intervencion_critica
2,avg_order_value,4.8455,estable
3,orders_12m,4.6211,estable
4,days_since_last_purchase,1.5032,estable
5,app_sessions_30d,1.2789,estable
6,discount_sensitivity,1.2022,estable
7,household_income,1.1317,estable
8,web_sessions_30d,1.1285,estable
9,basket_diversity,1.0179,estable


,columna,vif,nivel_alerta
0,orders_12m,1.5059,estable
1,days_since_last_purchase,1.5031,estable
2,app_sessions_30d,1.2699,estable
3,discount_sensitivity,1.1800,estable
4,web_sessions_30d,1.1224,estable
5,avg_order_value,1.0913,estable
6,household_income,1.0815,estable
7,basket_diversity,1.0157,estable


In [18]:
mcar_compatible = bool(missingness_audit['little_mcar'].iloc[0]['is_mcar_compatible'])

nps_gap = float(missing_profile.loc[missing_profile['variable'] == 'nps_score', 'churn_gap_pp'].iloc[0])

max_outlier_pct = float(winsorized_result['report']['pct_outliers_original'].max())

max_vif_post = float(vif_result['post_mitigation_report']['vif'].max())

display(Markdown(

    f"### Interpretacion de Hallazgos e Implicacion Operativa\n"

    f"- La prueba agregada de faltantes arroja compatibilidad MCAR = **{mcar_compatible}**, pero el faltante de NPS aumenta **{nps_gap:.2f} pp** entre clientes que abandonan. Operativamente, conviene tratar ese flag como ausencia informativa.\n"

    f"- El mayor bloque de outliers afecta aproximadamente **{max_outlier_pct:.2f}%** de una variable critica. Se aplico winsorizacion para conservar clientes valiosos y evitar que pocos extremos dominen la lectura del mercado.\n"

    f"- Tras la mitigacion, el VIF maximo cae a **{max_vif_post:.2f}**. Implicacion: los coeficientes del baseline logistico quedan en una zona mas estable y defendible para negocio."

))


### Interpretacion de Hallazgos e Implicacion Operativa
- La prueba agregada de faltantes arroja compatibilidad MCAR = **True**, pero el faltante de NPS aumenta **10.93 pp** entre clientes que abandonan. Operativamente, conviene tratar ese flag como ausencia informativa.
- El mayor bloque de outliers afecta aproximadamente **5.42%** de una variable critica. Se aplico winsorizacion para conservar clientes valiosos y evitar que pocos extremos dominen la lectura del mercado.
- Tras la mitigacion, el VIF maximo cae a **1.51**. Implicacion: los coeficientes del baseline logistico quedan en una zona mas estable y defendible para negocio.

## 4. Modelado y Parsimonia

**Por que**: en marketing aplicado importa explicar bien antes que perseguir complejidad si la ganancia marginal no compensa el coste operativo.

**Para que**: comparar un baseline interpretable con un challenger no lineal y promover la opcion simple cuando el gap de desempeno sea pequeno segun la tolerancia Occam del contrato.

In [9]:
logit_result = train_supervised_model(
    winsorized_df,
    target=CONTRACT['target'],
    features=MODEL_FEATURES,
    algorithm=CONTRACT['modeling']['baseline_algorithm'],
    test_size=CONTRACT['modeling']['test_size'],
    random_state=CONTRACT['modeling']['random_state'],
    probability_calibration='auto',
    verbose=False,
)
rf_result = train_supervised_model(
    winsorized_df,
    target=CONTRACT['target'],
    features=MODEL_FEATURES,
    algorithm=CONTRACT['modeling']['challenger_algorithm'],
    test_size=CONTRACT['modeling']['test_size'],
    random_state=CONTRACT['modeling']['random_state'],
    probability_calibration='auto',
    verbose=False,
)

model_comparison = pd.DataFrame([
    {'model': 'logistic', **logit_result['metrics'].iloc[0].to_dict()},
    {'model': 'random_forest', **rf_result['metrics'].iloc[0].to_dict()},
])
auc_gap = float(model_comparison.loc[model_comparison['model'] == 'random_forest', 'roc_auc'].iloc[0] - model_comparison.loc[model_comparison['model'] == 'logistic', 'roc_auc'].iloc[0])
champion_name = 'logistic' if auc_gap <= CONTRACT['modeling']['parsimony_auc_tolerance'] else 'random_forest'
champion_result = logit_result if champion_name == 'logistic' else rf_result

coef_report = pd.DataFrame({
    'feature': logit_result['pipeline'].named_steps['preprocessor'].get_feature_names_out(),
    'coefficient': logit_result['pipeline'].named_steps['model'].coef_.ravel(),
})
coef_report['abs_coefficient'] = coef_report['coefficient'].abs()
coef_report = coef_report.sort_values('abs_coefficient', ascending=False).head(12)
importance_report = champion_result['feature_importance'].head(12)

display(model_comparison)
if not champion_result['calibration_comparison'].empty:
    display(champion_result['calibration_comparison'])
display(coef_report)
display(importance_report)

,model,accuracy,precision,recall,f1,roc_auc,log_loss
0,logistic,0.7831,0.7625,0.7831,0.7658,0.7658,0.4679
1,random_forest,0.7862,0.7638,0.7862,0.7479,0.7690,0.4680


,method,brier_score,brier_score_delta,applied,selected_for_scoring,calibration_rows,note
0,raw,0.1519,0.0000,False,False,0,Probabilidad cruda del modelo base.
1,sigmoid,0.1501,0.0018,True,True,390,Calibrador sigmoid evaluado sobre el holdout.
2,isotonic,0.1572,-0.0053,False,False,390,Calibrador isotonic evaluado sobre el holdout.


,feature,coefficient,abs_coefficient
52,categorical__customer_segment_premium_families,-0.9323,0.9323
9,numeric__monetary_12m,-0.7343,0.7343
10,numeric__margin_12m,0.6583,0.6583
54,categorical__loyalty_tier_bronze,-0.5143,0.5143
43,categorical__acquisition_channel_paid_social,-0.5074,0.5074
22,numeric__subscription_opt_in,-0.4659,0.4659
46,categorical__preferred_channel_call_center,-0.4245,0.4245
30,numeric__nps_score_missing_flag,0.3726,0.3726
38,categorical__city_tier_tier_2,-0.3195,0.3195
8,numeric__orders_12m,-0.3148,0.3148


,feature,importance_mean,importance_std
15,monetary_12m,0.0605,0.0151
28,subscription_opt_in,0.0236,0.0063
18,nps_score,0.0213,0.0040
21,return_rate_12m,0.0212,0.0037
17,support_tickets_90d,0.0201,0.0041
16,margin_12m,0.0178,0.0098
4,customer_segment,0.0153,0.0069
36,nps_score_missing_flag,0.0134,0.0052
20,days_since_last_purchase,0.0101,0.0038
14,orders_12m,0.0093,0.0054


In [10]:
logit_auc = float(logit_result['metrics'].iloc[0]['roc_auc'])
rf_auc = float(rf_result['metrics'].iloc[0]['roc_auc'])
display(Markdown(
    f"### Interpretacion de Hallazgos e Implicacion Operativa\n"
    f"- La logistica logra un ROC AUC de **{logit_auc:.4f}** y el random forest **{rf_auc:.4f}**.\n"
    f"- El gap de desempeno es **{auc_gap:.4f}**, frente a una tolerancia Occam de **{CONTRACT['modeling']['parsimony_auc_tolerance']:.4f}**.\n"
    f"- Champion seleccionado: **{champion_name}**. Implicacion: la operacion puede usar un score interpretable siempre que el challenger no aporte una mejora material. Esto simplifica explicacion a marketing, compliance y equipos CRM."
))

### Interpretacion de Hallazgos e Implicacion Operativa
- La logistica logra un ROC AUC de **0.7658** y el random forest **0.7690**.
- El gap de desempeno es **0.0032**, frente a una tolerancia Occam de **0.0300**.
- Champion seleccionado: **logistic**. Implicacion: la operacion puede usar un score interpretable siempre que el challenger no aporte una mejora material. Esto simplifica explicacion a marketing, compliance y equipos CRM.

## 5. Capa de Marketing Engineering y CLV

**Por que**: priorizar solo por probabilidad de churn puede desperdiciar presupuesto en clientes de bajo valor y dejar fuera cuentas con alto margen futuro.

**Para que**: estimar CLV futuro con un enfoque probabilistico, convertir el riesgo en valor esperado neto y encontrar el umbral de activacion que maximiza ROI.

In [11]:
frequency = winsorized_df['clv_frequency'].astype(float)
recency = winsorized_df['clv_recency_days'].astype(float)
T = winsorized_df['clv_T_days'].astype(float)
monetary = winsorized_df['clv_monetary_avg'].astype(float)
valid_mask = (frequency >= 0) & (recency >= 0) & (T > 0) & (recency <= T)
repeat_mask = valid_mask & (frequency > 0) & (monetary > 0)

predicted_tx_6m = pd.Series(np.zeros(len(winsorized_df)), index=winsorized_df.index, dtype=float)
expected_avg_margin = pd.Series(monetary.copy(), index=winsorized_df.index, dtype=float)
clv_engine = 'bg_nbd'
ggf_engine = 'empirical_margin'

try:
    bgf = BetaGeoFitter(penalizer_coef=float(CONTRACT['clv']['bgf_penalizer']))
    bgf.fit(frequency[valid_mask], recency[valid_mask], T[valid_mask])
    predicted_tx_6m.loc[valid_mask] = bgf.conditional_expected_number_of_purchases_up_to_time(
        float(CONTRACT['clv']['horizon_months']) * 30,
        frequency[valid_mask],
        recency[valid_mask],
        T[valid_mask],
    )
except Exception:
    clv_engine = 'frequency_heuristic'
    predicted_tx_6m.loc[valid_mask] = (frequency[valid_mask] / np.maximum(T[valid_mask], 1.0)) * float(CONTRACT['clv']['horizon_months']) * 30

if repeat_mask.sum() >= 30:
    try:
        ggf = GammaGammaFitter(penalizer_coef=float(CONTRACT['clv']['ggf_penalizer']))
        ggf.fit(frequency[repeat_mask], monetary[repeat_mask])
        expected_avg_margin.loc[repeat_mask] = ggf.conditional_expected_average_profit(
            frequency[repeat_mask],
            monetary[repeat_mask],
        )
        ggf_engine = 'gamma_gamma'
    except Exception:
        ggf_engine = 'empirical_margin'

raw_probability = champion_result['pipeline'].predict_proba(winsorized_df[MODEL_FEATURES])[:, 1]
scored_probability = apply_calibration(champion_result, raw_probability)
clv_6m_margin = (predicted_tx_6m * expected_avg_margin).clip(lower=0)

scoring_frame = pd.DataFrame({
    'customer_id': winsorized_df['customer_id'],
    'predicted_churn_probability': scored_probability,
    'predicted_transactions_6m': predicted_tx_6m.round(3),
    'predicted_clv_6m_margin': clv_6m_margin.round(2),
})
scoring_frame['expected_net_value'] = (
    scoring_frame['predicted_churn_probability']
    * float(CONTRACT['contact_policy']['retention_success_rate'])
    * scoring_frame['predicted_clv_6m_margin']
    - float(CONTRACT['contact_policy']['contact_cost'])
    - float(CONTRACT['contact_policy']['offer_cost'])
).round(2)

threshold_report = build_threshold_report(scoring_frame, CONTRACT['contact_policy'], CONTRACT['modeling'])
best_threshold = threshold_report.iloc[0]
clv_summary = pd.DataFrame([
    {
        'clv_engine': clv_engine,
        'margin_engine': ggf_engine,
        'mean_predicted_transactions_6m': round(float(scoring_frame['predicted_transactions_6m'].mean()), 3),
        'mean_predicted_clv_6m_margin': round(float(scoring_frame['predicted_clv_6m_margin'].mean()), 2),
        'max_expected_net_value': round(float(scoring_frame['expected_net_value'].max()), 2),
    }
])

display(clv_summary)
display(threshold_report.head(10))

fig, ax = plt.subplots(figsize=(8, 4))
threshold_report.sort_values('threshold').plot(
    x='threshold',
    y='total_expected_net_value',
    kind='line',
    marker='o',
    ax=ax,
    color='#0F766E',
)
ax.set_title('Valor esperado neto por umbral de activacion')
ax.set_ylabel('Total expected net value')
ax.axvline(best_threshold['threshold'], color='#EA580C', linestyle='--', label='best threshold')
ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

,clv_engine,margin_engine,mean_predicted_transactions_6m,mean_predicted_clv_6m_margin,max_expected_net_value
0,bg_nbd,gamma_gamma,0.9930,17.6100,8.5000


,threshold,selected_customers,contact_share,total_expected_net_value,average_expected_net_value,roi
0,0.1500,24,0.0092,50.9400,2.1200,0.2908
1,0.2000,20,0.0077,40.4800,2.0200,0.2773
2,0.2500,19,0.0073,38.4800,2.0300,0.2774
3,0.3000,18,0.0069,38.0400,2.1100,0.2895
4,0.3500,16,0.0062,36.7000,2.2900,0.3142
5,0.4000,16,0.0062,36.7000,2.2900,0.3142
6,0.4500,16,0.0062,36.7000,2.2900,0.3142
7,0.5000,11,0.0042,28.4700,2.5900,0.3545
8,0.5500,9,0.0035,25.8400,2.8700,0.3933
9,0.6000,8,0.0031,24.2400,3.0300,0.4151


<Figure size 800x400 with 1 Axes>

In [12]:
display(Markdown(
    f"### Interpretacion de Hallazgos e Implicacion Operativa\n"
    f"- El motor de CLV utiliza **{clv_engine}** para frecuencia futura y **{ggf_engine}** para margen medio por compra.\n"
    f"- El umbral que maximiza valor esperado neto es **{best_threshold['threshold']:.2f}**, con **{int(best_threshold['selected_customers'])} clientes** y un total esperado de **{float(best_threshold['total_expected_net_value']):,.2f}**.\n"
    f"- Implicacion operativa: marketing no debe contactar a todo cliente riesgoso. Conviene priorizar solo la franja donde riesgo y CLV esperado superan el coste de accion."
))

### Interpretacion de Hallazgos e Implicacion Operativa
- El motor de CLV utiliza **bg_nbd** para frecuencia futura y **gamma_gamma** para margen medio por compra.
- El umbral que maximiza valor esperado neto es **0.15**, con **24 clientes** y un total esperado de **50.94**.
- Implicacion operativa: marketing no debe contactar a todo cliente riesgoso. Conviene priorizar solo la franja donde riesgo y CLV esperado superan el coste de accion.

## 6. Activacion Tactica: Next Best Action

**Por que**: un score sin accion recomendada obliga al equipo CRM a improvisar canal y oferta, degradando la conversion comercial.

**Para que**: traducir el riesgo y el valor esperado a un scorecard operativo con NBA, canal recomendado y oferta concreta por cliente.

In [13]:
activation_base = winsorized_df[[
    'customer_id',
    'preferred_channel',
    'customer_segment',
    'loyalty_tier',
    'discount_sensitivity',
    'support_tickets_90d',
    'premium_pack_adoption',
    'nps_score_missing_flag',
]].copy()

scorecard = activation_base.merge(scoring_frame, on='customer_id', how='left')
max_contacts = int(len(scorecard) * float(CONTRACT['contact_policy']['max_contact_share']))
selected_scorecard = scorecard.loc[
    (scorecard['predicted_churn_probability'] >= best_threshold['threshold'])
    & (scorecard['expected_net_value'] > 0)
].sort_values(['expected_net_value', 'predicted_churn_probability'], ascending=False).head(max_contacts).copy()

selected_scorecard['risk_tier'] = pd.cut(
    selected_scorecard['predicted_churn_probability'],
    bins=[-np.inf, CONTRACT['nba_policy']['medium_threshold'], CONTRACT['nba_policy']['high_threshold'], CONTRACT['nba_policy']['critical_threshold'], np.inf],
    labels=['Monitor', 'Medio', 'Alto', 'Critico'],
)
selected_scorecard['recommended_channel'] = selected_scorecard.apply(assign_channel, axis=1)
selected_scorecard['offer'] = selected_scorecard.apply(assign_offer, axis=1)
selected_scorecard['next_best_action'] = selected_scorecard.apply(assign_nba, axis=1)

action_mix = selected_scorecard.groupby(['risk_tier', 'next_best_action', 'recommended_channel'], observed=False).size().reset_index(name='customers').sort_values('customers', ascending=False)
shortlist_columns = [
    'customer_id',
    'risk_tier',
    'predicted_churn_probability',
    'predicted_clv_6m_margin',
    'expected_net_value',
    'next_best_action',
    'recommended_channel',
    'offer',
]

scorecard.to_csv(PROCESSED_DIR / 'retail_scorecard_full.csv', index=False)
selected_scorecard.to_csv(PROCESSED_DIR / 'retail_scorecard_shortlist.csv', index=False)
threshold_report.to_csv(PROCESSED_DIR / 'retail_threshold_optimization.csv', index=False)

display(action_mix.head(12))
display(selected_scorecard[shortlist_columns].head(20))

,risk_tier,next_best_action,recommended_channel,customers
6,Monitor,recuperacion de experiencia,sms,5
19,Medio,recuperacion de experiencia,whatsapp,3
30,Alto,recuperacion de experiencia,sms,2
31,Alto,recuperacion de experiencia,whatsapp,2
43,Critico,recuperacion de experiencia,whatsapp,2
41,Critico,recuperacion de experiencia,email,2
4,Monitor,recuperacion de experiencia,app_push,1
0,Monitor,recordatorio de recompra,app_push,1
17,Medio,recuperacion de experiencia,email,1
18,Medio,recuperacion de experiencia,sms,1


,customer_id,risk_tier,predicted_churn_probability,predicted_clv_6m_margin,expected_net_value,next_best_action,recommended_channel,offer
1046,CUST_01047,Monitor,0.1927,241.1500,8.5000,recuperacion de experiencia,app_push,upgrade bundle premium con envio gratis
2475,CUST_02476,Critico,0.6841,65.6700,7.9700,recuperacion de experiencia,email,upgrade bundle premium con envio gratis
766,CUST_00767,Critico,0.7370,50.9400,5.4600,recuperacion de experiencia,app_push,bono de servicio y reposicion prioritaria
267,CUST_00268,Alto,0.6159,55.9700,4.4200,recuperacion de experiencia,sms,bono de servicio y reposicion prioritaria
2279,CUST_02280,Alto,0.6075,53.7500,3.8000,recuperacion de experiencia,sms,bono de servicio y reposicion prioritaria
137,CUST_00138,Medio,0.4991,64.8000,3.7000,recuperacion de experiencia,whatsapp,bono de servicio y reposicion prioritaria
511,CUST_00512,Medio,0.4881,59.6000,2.5900,recuperacion de experiencia,whatsapp,bono de servicio y reposicion prioritaria
2114,CUST_02115,Monitor,0.2264,120.8300,2.0000,recuperacion de experiencia,sms,cupon 15% en segunda compra
1099,CUST_01100,Alto,0.5687,46.0500,1.6000,recuperacion de experiencia,app_push,bono de servicio y reposicion prioritaria
1517,CUST_01518,Medio,0.4932,53.0500,1.6000,recuperacion de experiencia,sms,bono de servicio y reposicion prioritaria


In [14]:
critical_clients = int(selected_scorecard['risk_tier'].eq('Critico').sum())
display(Markdown(
    f"### Interpretacion de Hallazgos e Implicacion Operativa\n"
    f"- El scorecard final prioriza **{len(selected_scorecard)} clientes** con valor esperado neto positivo y limita la carga comercial al maximo definido por gobernanza.\n"
    f"- Dentro de la cartera priorizada hay **{critical_clients} clientes criticos**. Estos casos son los que justifican canales mas caros o de mayor friccion humana.\n"
    f"- El notebook exporta `retail_scorecard_full.csv`, `retail_scorecard_shortlist.csv` y `retail_threshold_optimization.csv` en `data/processed/`. Implicacion: el analisis ya queda listo para una campaña CRM o una discusion de ROI con marketing."
))

### Interpretacion de Hallazgos e Implicacion Operativa
- El scorecard final prioriza **24 clientes** con valor esperado neto positivo y limita la carga comercial al maximo definido por gobernanza.
- Dentro de la cartera priorizada hay **6 clientes criticos**. Estos casos son los que justifican canales mas caros o de mayor friccion humana.
- El notebook exporta `retail_scorecard_full.csv`, `retail_scorecard_shortlist.csv` y `retail_threshold_optimization.csv` en `data/processed/`. Implicacion: el analisis ya queda listo para una campaña CRM o una discusion de ROI con marketing.